# WRDS Extract 04: Sanity Checks

Run `01_wrds_extract_membership.ipynb`, `02_wrds_extract_crsp_panel.ipynb`, and `03_wrds_extract_ff3.ipynb` first so the three raw parquet files already exist. This notebook checks coverage, duplicate keys, missingness, and monthly count consistency, then writes summary CSV/TXT/PNG outputs.

In [ ]:
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
indir = Path("data/raw")
outdir = Path("data/sanity_checks")

membership_file = indir / "sp500_membership_monthly.parquet"
crsp_file = indir / "crsp_monthly_sp500_panel.parquet"
ff3_file = indir / "ff3_monthly.parquet"

outdir.mkdir(parents=True, exist_ok=True)

membership_file, crsp_file, ff3_file, outdir

## Helper Functions

In [ ]:
CRSP_MISSINGNESS_COLUMNS = [
    "ret",
    "retx",
    "dlret",
    "prc",
    "shrout",
    "vol",
    "me",
    "retadj",
]
FF3_MISSINGNESS_COLUMNS = ["mktrf", "smb", "hml", "rf"]


def read_parquet_file(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")
    return pd.read_parquet(path)


def require_columns(df: pd.DataFrame, columns: Iterable[str], dataset_name: str) -> None:
    missing = sorted(set(columns).difference(df.columns))
    if missing:
        raise ValueError(
            f"{dataset_name} is missing required columns: {', '.join(missing)}"
        )


def normalize_month_end(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    require_columns(df, ["month_end"], dataset_name)
    result = df.copy()
    result["month_end"] = pd.to_datetime(result["month_end"], errors="coerce")
    if result["month_end"].isna().any():
        bad_rows = int(result["month_end"].isna().sum())
        raise ValueError(f"{dataset_name} has {bad_rows:,} invalid month_end values.")
    return result


def format_date(value: pd.Timestamp | None) -> str:
    if value is None or pd.isna(value):
        return "NA"
    return pd.Timestamp(value).date().isoformat()


def date_min_max(df: pd.DataFrame) -> tuple[pd.Timestamp | None, pd.Timestamp | None]:
    if df.empty:
        return None, None
    return df["month_end"].min(), df["month_end"].max()


def count_by_month(
    df: pd.DataFrame,
    count_column_name: str,
    *,
    unique_permnos: bool = True,
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(
            {
                "month_end": pd.Series(dtype="datetime64[ns]"),
                count_column_name: pd.Series(dtype="int64"),
            }
        )
    if unique_permnos:
        counts = df.groupby("month_end")["permno"].nunique()
    else:
        counts = df.groupby("month_end").size()
    return (
        counts.rename(count_column_name)
        .reset_index()
        .sort_values("month_end")
        .reset_index(drop=True)
    )


def count_summary_stats(counts: pd.Series) -> pd.Series:
    return counts.describe(percentiles=[0.25, 0.5, 0.75])


def format_count_stats(stats: pd.Series) -> str:
    if stats.empty:
        return "count=0"
    return (
        f"count={stats['count']:.0f}, mean={stats['mean']:.2f}, "
        f"std={stats['std']:.2f}, min={stats['min']:.2f}, "
        f"p25={stats['25%']:.2f}, p50={stats['50%']:.2f}, "
        f"p75={stats['75%']:.2f}, max={stats['max']:.2f}"
    )


def missingness_summary(
    df: pd.DataFrame, columns: list[str], dataset_name: str
) -> pd.DataFrame:
    require_columns(df, columns, dataset_name)
    total_rows = len(df)
    rows = []
    for col in columns:
        missing_count = int(df[col].isna().sum())
        missing_rate = missing_count / total_rows if total_rows else 0.0
        rows.append(
            {
                "dataset": dataset_name,
                "column": col,
                "missing_count": missing_count,
                "total_rows": total_rows,
                "missing_rate": missing_rate,
            }
        )
    return pd.DataFrame(rows)


def duplicate_key_summary(df: pd.DataFrame, keys: list[str]) -> tuple[bool, int, int]:
    duplicate_mask = df.duplicated(keys, keep=False)
    duplicate_rows = int(duplicate_mask.sum())
    duplicate_keys = int(df.loc[duplicate_mask, keys].drop_duplicates().shape[0])
    return duplicate_rows > 0, duplicate_rows, duplicate_keys


def compare_monthly_counts(
    membership_counts: pd.DataFrame, crsp_counts: pd.DataFrame
) -> pd.DataFrame:
    comparison = crsp_counts.merge(membership_counts, on="month_end", how="outer")
    comparison = comparison.sort_values("month_end").reset_index(drop=True)
    comparison["crsp_count"] = comparison["crsp_count"].fillna(0).astype(int)
    comparison["membership_count"] = comparison["membership_count"].fillna(0).astype(int)
    comparison["crsp_minus_membership"] = (
        comparison["crsp_count"] - comparison["membership_count"]
    )
    return comparison


def factor_coverage_summary(
    crsp: pd.DataFrame, ff3: pd.DataFrame
) -> tuple[int, list[pd.Timestamp]]:
    crsp_months = crsp[["month_end"]].dropna().drop_duplicates()
    ff_months = ff3[["month_end"]].dropna().drop_duplicates()
    factor_months = ff_months.assign(has_factor_data=True)
    coverage = crsp_months.merge(factor_months, on="month_end", how="left")
    missing_factor_months = sorted(
        coverage.loc[coverage["has_factor_data"].isna(), "month_end"]
    )
    matched_months = int(coverage["has_factor_data"].fillna(False).sum())
    return matched_months, missing_factor_months

In [ ]:
def build_coverage_summary_text(
    membership: pd.DataFrame,
    crsp: pd.DataFrame,
    ff3: pd.DataFrame,
    membership_counts: pd.DataFrame,
    count_comparison: pd.DataFrame,
    crsp_has_duplicates: bool,
    crsp_duplicate_rows: int,
    crsp_duplicate_keys: int,
    ff3_has_duplicate_months: bool,
    ff3_duplicate_rows: int,
    ff3_duplicate_months: int,
    matched_factor_months: int,
    missing_factor_months: list[pd.Timestamp],
) -> str:
    membership_min, membership_max = date_min_max(membership)
    crsp_min, crsp_max = date_min_max(crsp)
    ff3_min, ff3_max = date_min_max(ff3)
    membership_stats = count_summary_stats(membership_counts["membership_count"])
    crsp_stats = count_summary_stats(count_comparison["crsp_count"])
    diff_stats = count_summary_stats(count_comparison["crsp_minus_membership"])

    lines = [
        "WRDS extract sanity check",
        "",
        "Membership",
        f"- month_end range: {format_date(membership_min)} to {format_date(membership_max)}",
        f"- rows: {len(membership):,}",
        f"- unique permnos: {membership['permno'].nunique():,}",
        f"- monthly constituent count stats: {format_count_stats(membership_stats)}",
        "",
        "CRSP panel",
        f"- month_end range: {format_date(crsp_min)} to {format_date(crsp_max)}",
        f"- rows: {len(crsp):,}",
        f"- unique permnos: {crsp['permno'].nunique():,}",
        f"- duplicate (permno, month_end) keys: {'YES' if crsp_has_duplicates else 'NO'}",
        f"- duplicate rows involved: {crsp_duplicate_rows:,}",
        f"- duplicate key count: {crsp_duplicate_keys:,}",
        f"- monthly CRSP count stats: {format_count_stats(crsp_stats)}",
        f"- CRSP minus membership count stats: {format_count_stats(diff_stats)}",
        "",
        "Fama-French factors",
        f"- month_end range: {format_date(ff3_min)} to {format_date(ff3_max)}",
        f"- rows: {len(ff3):,}",
        f"- unique month_end: {'NO' if ff3_has_duplicate_months else 'YES'}",
        f"- duplicate month rows involved: {ff3_duplicate_rows:,}",
        f"- duplicate month count: {ff3_duplicate_months:,}",
        "",
        "CRSP and FF3 merge coverage",
        f"- matched CRSP months with factor rows: {matched_factor_months:,}",
        f"- any CRSP month missing factor data: {'YES' if missing_factor_months else 'NO'}",
    ]

    if missing_factor_months:
        missing_str = ", ".join(format_date(month) for month in missing_factor_months)
        lines.append(f"- CRSP months missing factors: {missing_str}")

    return "\n".join(lines) + "\n"


def save_line_plot(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    ylabel: str,
    output_path: Path,
) -> None:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print(f"Skipping plot because matplotlib is not installed: {output_path.name}")
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df[x_col], df[y_col], linewidth=1.6)
    ax.set_title(title)
    ax.set_xlabel("Month end")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.show()

## Load Extracts

In [ ]:
membership = normalize_month_end(read_parquet_file(membership_file), "membership")
crsp = normalize_month_end(read_parquet_file(crsp_file), "crsp")
ff3 = normalize_month_end(read_parquet_file(ff3_file), "ff3")

require_columns(membership, ["permno", "month_end"], "membership")
require_columns(crsp, ["permno", "month_end"], "crsp")
require_columns(ff3, ["month_end"], "ff3")

summary = pd.DataFrame(
    [
        {
            "dataset": "membership",
            "rows": len(membership),
            "unique_permnos": membership["permno"].nunique(),
            "month_start": date_min_max(membership)[0],
            "month_end": date_min_max(membership)[1],
        },
        {
            "dataset": "crsp",
            "rows": len(crsp),
            "unique_permnos": crsp["permno"].nunique(),
            "month_start": date_min_max(crsp)[0],
            "month_end": date_min_max(crsp)[1],
        },
        {
            "dataset": "ff3",
            "rows": len(ff3),
            "unique_permnos": pd.NA,
            "month_start": date_min_max(ff3)[0],
            "month_end": date_min_max(ff3)[1],
        },
    ]
)

summary

## Monthly Coverage

In [ ]:
membership_counts = count_by_month(membership, "membership_count")
crsp_counts = count_by_month(crsp, "crsp_count")
count_comparison = compare_monthly_counts(membership_counts, crsp_counts)

display(count_comparison.head())
display(count_comparison.tail())
count_comparison[["membership_count", "crsp_count", "crsp_minus_membership"]].describe()

In [ ]:
largest_count_gaps = count_comparison.assign(
    abs_gap=count_comparison["crsp_minus_membership"].abs()
).sort_values(["abs_gap", "month_end"], ascending=[False, True])

largest_count_gaps.head(20)

## Missingness

In [ ]:
missingness = pd.concat(
    [
        missingness_summary(crsp, CRSP_MISSINGNESS_COLUMNS, "crsp"),
        missingness_summary(ff3, FF3_MISSINGNESS_COLUMNS, "ff3"),
    ],
    ignore_index=True,
)

missingness

## Duplicate Keys

In [ ]:
crsp_has_duplicates, crsp_duplicate_rows, crsp_duplicate_keys = duplicate_key_summary(
    crsp, ["permno", "month_end"]
)
ff3_has_duplicate_months, ff3_duplicate_rows, ff3_duplicate_months = duplicate_key_summary(
    ff3, ["month_end"]
)

duplicate_summary = pd.DataFrame(
    [
        {
            "dataset": "crsp",
            "key": "permno, month_end",
            "has_duplicates": crsp_has_duplicates,
            "duplicate_rows": crsp_duplicate_rows,
            "duplicate_keys": crsp_duplicate_keys,
        },
        {
            "dataset": "ff3",
            "key": "month_end",
            "has_duplicates": ff3_has_duplicate_months,
            "duplicate_rows": ff3_duplicate_rows,
            "duplicate_keys": ff3_duplicate_months,
        },
    ]
)

duplicate_summary

In [ ]:
if crsp_has_duplicates:
    display(
        crsp.loc[crsp.duplicated(["permno", "month_end"], keep=False)]
        .sort_values(["permno", "month_end"])
        .head(50)
    )
else:
    print("No duplicate CRSP (permno, month_end) keys found.")

if ff3_has_duplicate_months:
    display(
        ff3.loc[ff3.duplicated(["month_end"], keep=False)]
        .sort_values("month_end")
        .head(50)
    )
else:
    print("No duplicate FF3 month_end keys found.")

## FF3 Coverage

In [ ]:
matched_factor_months, missing_factor_months = factor_coverage_summary(crsp, ff3)

factor_coverage = pd.DataFrame(
    [
        {
            "crsp_distinct_months": crsp["month_end"].nunique(),
            "ff3_distinct_months": ff3["month_end"].nunique(),
            "matched_crsp_months_with_ff3": matched_factor_months,
            "missing_crsp_months_with_ff3": len(missing_factor_months),
        }
    ]
)

display(factor_coverage)

if missing_factor_months:
    display(pd.DataFrame({"missing_month_end": missing_factor_months}))
else:
    print("All CRSP months have FF3 factor rows.")

## Save Tables and Report

In [ ]:
membership_counts.to_csv(outdir / "membership_monthly_counts.csv", index=False)
count_comparison.to_csv(outdir / "crsp_monthly_counts.csv", index=False)
missingness.to_csv(outdir / "missingness_summary.csv", index=False)
duplicate_summary.to_csv(outdir / "duplicate_key_summary.csv", index=False)
factor_coverage.to_csv(outdir / "factor_coverage_summary.csv", index=False)

coverage_summary_text = build_coverage_summary_text(
    membership=membership,
    crsp=crsp,
    ff3=ff3,
    membership_counts=membership_counts,
    count_comparison=count_comparison,
    crsp_has_duplicates=crsp_has_duplicates,
    crsp_duplicate_rows=crsp_duplicate_rows,
    crsp_duplicate_keys=crsp_duplicate_keys,
    ff3_has_duplicate_months=ff3_has_duplicate_months,
    ff3_duplicate_rows=ff3_duplicate_rows,
    ff3_duplicate_months=ff3_duplicate_months,
    matched_factor_months=matched_factor_months,
    missing_factor_months=missing_factor_months,
)
(outdir / "coverage_summary.txt").write_text(coverage_summary_text, encoding="utf-8")

print(coverage_summary_text)

## Plots

In [ ]:
save_line_plot(
    membership_counts,
    "month_end",
    "membership_count",
    "Monthly S&P 500 Constituent Count",
    "Constituent count",
    outdir / "membership_monthly_count.png",
)

save_line_plot(
    count_comparison,
    "month_end",
    "crsp_count",
    "Monthly CRSP S&P 500 Panel Count",
    "CRSP stock count",
    outdir / "crsp_monthly_count.png",
)

save_line_plot(
    count_comparison,
    "month_end",
    "crsp_minus_membership",
    "CRSP Count Minus Membership Count",
    "CRSP count - membership count",
    outdir / "crsp_minus_membership_count.png",
)

## Final Checks

In [ ]:
max_abs_count_gap = (
    int(count_comparison["crsp_minus_membership"].abs().max())
    if not count_comparison.empty
    else 0
)

final_checks = pd.DataFrame(
    [
        {"check": "CRSP duplicate (permno, month_end) keys", "value": crsp_duplicate_keys},
        {"check": "FF3 duplicate month_end keys", "value": ff3_duplicate_months},
        {"check": "CRSP months missing FF3 factor data", "value": len(missing_factor_months)},
        {"check": "Max absolute CRSP-minus-membership count gap", "value": max_abs_count_gap},
    ]
)

print(f"Outputs saved to: {outdir}")
final_checks